<a href="https://colab.research.google.com/github/seojun779/web_2026_bigdatacomputing/blob/main/20231338_%EC%9D%B4%EC%84%9C%EC%A4%80_%EA%B8%B0%EB%A7%90%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# 필수 패키지 설치
!pip install streamlit pyngrok matplotlib seaborn

# ngrok 인증 토큰 설정 ($YOUR_AUTHTOKEN)
# 토큰 확인 주소: https://dashboard.ngrok.com/get-started/your-authtoken
import pyngrok.ngrok as ngrok
# ngrok.set_auth_token("3EzaYVzYZX36N0Xqpe2aVjrxrEN_4pDduoGZ41nLKbMpgLtyG")

In [15]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error

# ============================================================
# 1. WHO 기대수명 데이터 불러오기 및 전처리
# ============================================================
url = "https://github.com/dongupak/DataML/raw/main/csv/life_expectancy.csv"
df = pd.read_csv(url)

# 컬럼명 앞뒤 공백 제거 및 결측치 제거
df.columns = df.columns.str.strip()
df = df.dropna()

# 특성 선택 (Schooling 제외, 최소 3개 이상)
features = ['Adult mortality', 'BMI', 'GDP', 'Alcohol']
target = 'Life expectancy'

X = df[features]
y = df[target]

# ============================================================
# 2. Train/Test 분리 및 소규모 훈련 샘플링 (50개)
# ============================================================
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

np.random.seed(42)
sample_index = np.random.choice(X_train_full.index, size=50, replace=False)
X_train = X_train_full.loc[sample_index]
y_train = y_train_full.loc[sample_index]

# ============================================================
# 3. 파이프라인 구축 (3종 모델)
# ============================================================
# 데이터가 50개로 적을 때 3차(degree=3)는 과적합이 매우 심하므로 2차로 조절하는 것이 안정적입니다.
DEGREE = 2

models = {
    "Linear": Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', LinearRegression())
    ]),
    "Poly": Pipeline([
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(degree=DEGREE)),
        ('regressor', LinearRegression())
    ]),
    "Ridge": Pipeline([
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(degree=DEGREE)),
        ('regressor', Ridge(alpha=1.0))
    ])
}

# ============================================================
# 4. 모델 학습 및 평가 (반복문 및 로직 최적화)
# ============================================================
performance_results = []
trained_models = {}

for name, model in models.items():
    # 모델 학습 및 저장
    model.fit(X_train, y_train)
    trained_models[name] = model

    # 예측 수행
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    # 모델 복잡도(특성 수) 자동 추출 구조 개선
    if 'poly' in model.named_steps:
        complexity = model.named_steps['poly'].n_output_features_
    else:
        complexity = len(features)

    # 결과 데이터 축적
    performance_results.append({
        "Model": name,
        "Train R²": r2_score(y_train, train_pred),
        "Test R²": r2_score(y_test, test_pred),
        "Train MSE": mean_squared_error(y_train, train_pred),
        "Test MSE": mean_squared_error(y_test, test_pred),
        "Complexity": complexity
    })

# DataFrame 변환 및 출력 규칙 설정
performance_df = pd.DataFrame(performance_results)

print("=== 모델 학습 완료 및 최종 성능 ===")
print(performance_df.to_string(index=False)) # 인덱스 없이 깔끔하게 표 형태로 출력

# ============================================================
# 5. Streamlit용 파일 저장
# ============================================================
payload = {
    "models": trained_models,
    "performance": performance_df,
    "features": features,
    "X_test": X_test,
    "y_test": y_test
}

joblib.dump(payload, "life_expectancy_models.pkl")
print("\n'life_expectancy_models.pkl' 저장 완료")

=== 모델 학습 완료 및 최종 성능 ===
 Model  Train R²  Test R²  Train MSE  Test MSE  Complexity
Linear  0.733584 0.579432  29.216463 29.869771           4
  Poly  0.805545 0.093759  21.324809 64.363495          15
 Ridge  0.797948 0.549617  22.158020 31.987328          15

'life_expectancy_models.pkl' 저장 완료


In [16]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

# 페이지 설정
st.set_page_config(
    page_title="WHO 기대수명 예측 시스템",
    layout="wide"
)

st.title("🌍 WHO 기대수명 예측 머신러닝 웹 서비스")
st.write("Linear, Polynomial, Ridge 회귀 모델을 비교하고 기대수명을 실시간으로 예측합니다.")

# 저장된 파일 불러오기
payload = joblib.load("life_expectancy_models.pkl")
models = payload["models"]
performance_df = payload["performance"]
features = payload["features"]
X_test = payload["X_test"]

# ------------------------------------------------------------
# [조건 3] 모델 성능 비교 화면
# ------------------------------------------------------------
st.header("📊 모델 성능 비교")

# 1. 성능 평가지표 테이블 출력
st.dataframe(
    performance_df,
    use_container_width=True,
    hide_index=True
)

# 2. Test R2 점수 비교 막대그래프 시각화
st.subheader("📈 Test R² 점수 비교")
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#abcdef', '#ff9999', '#99ff99']
bars = ax.bar(performance_df["Model"], performance_df["Test R²"], color=colors, edgecolor='black')

ax.set_ylabel("Test R²")
ax.set_title("Model Comparison (Test R²)", fontsize=14)
ax.set_ylim(min(performance_df["Test R²"].min() - 0.2, 0), 1.1)

# 그래프 위에 수치 표시
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f'{yval:.4f}', ha='center', va='bottom', fontweight='bold')

st.pyplot(fig)

# ------------------------------------------------------------
# [조건 4] 사이드바 UI 및 실시간 예측 구성
# ------------------------------------------------------------
st.sidebar.header("🔧 독립변수(Features) 입력값 설정")

input_values = {}
for feature in features:
    min_val = float(X_test[feature].min())
    max_val = float(X_test[feature].max())
    mean_val = float(X_test[feature].mean())

    input_values[feature] = st.sidebar.slider(
        label=feature,
        min_value=min_val,
        max_value=max_val,
        value=mean_val
    )

st.header("🎯 실시간 기대수명 예측")
selected_model = st.selectbox(
    "예측에 사용할 머신러닝 모델을 선택하세요:",
    ["Linear", "Poly", "Ridge"]
)

# 예측 수행
input_df = pd.DataFrame([input_values])
model = models[selected_model]
prediction = model.predict(input_df)[0]

# 결과 큰 글씨 출력
st.metric(
    label=f"[{selected_model} 모델] 예측 기대수명",
    value=f"{prediction:.2f} 세"
)

# 입력값 요약 제공
st.subheader("💡 현재 입력된 데이터 상스 정보")
st.table(input_df)

Overwriting app.py


In [17]:
import os
import time
from pyngrok import ngrok

# ============================================================
# 1. 기존 ngrok 터널 종료
# ============================================================

ngrok.kill()

# ============================================================
# 2. ngrok 인증 토큰 등록
# ============================================================

ngrok.set_auth_token(
    "3EzaYVzYZX36N0Xqpe2aVjrxrEN_4pDduoGZ41nLKbMpgLtyG"
)

print("✅ ngrok 토큰 등록 완료")

# ============================================================
# 3. Streamlit 실행
# ============================================================

os.system(
    "nohup streamlit run app.py "
    "--server.port 8501 "
    "> streamlit.log 2>&1 &"
)

print("🚀 Streamlit 실행 중...")

# Streamlit이 켜질 시간을 조금 기다림
time.sleep(5)

# ============================================================
# 4. ngrok 터널 생성
# ============================================================

try:
    tunnel = ngrok.connect(8501)

    print("=" * 60)
    print("🎉 Streamlit 웹 서비스 배포 성공!")
    print("외부 접속 주소:")
    print(tunnel.public_url)
    print("=" * 60)

except Exception as e:
    print("❌ ngrok 연결 실패")
    print(e)

✅ ngrok 토큰 등록 완료
🚀 Streamlit 실행 중...
🎉 Streamlit 웹 서비스 배포 성공!
외부 접속 주소:
https://sequester-stereo-deflator.ngrok-free.dev
